# Notebook 01 — IoT 센서 데이터 탐색적 분석 (EDA)

**목표:** 정상/이상 패턴 데이터의 분포를 이해하고, 이상탐지 모델에 사용할 피처를 선정한다.

**데이터:** 온도(°C) / 진동(mm/s) / 전류(A) 3채널 센서
- 정상: 10,000건
- 이상: 500건 (과열 / 과진동 / 전류 스파이크 3종)

In [ ]:
# 한글 폰트 설정
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import pandas as pd
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('라이브러리 로드 완료')

In [ ]:
# 데이터 로드
df = pd.read_csv('../data/raw/sensor_data.csv', parse_dates=['timestamp'])
df['label_name'] = df['label'].map({0: '정상', 1: '이상'})

print(f'전체 데이터: {len(df):,}건')
print(f'정상: {(df.label==0).sum():,}건 ({(df.label==0).mean()*100:.1f}%)')
print(f'이상: {(df.label==1).sum():,}건 ({(df.label==1).mean()*100:.1f}%)')
df.head()

## 1. 기초 통계 — 정상 vs 이상 비교

In [ ]:
features = ['temperature', 'vibration', 'current']
labels   = ['온도 (°C)', '진동 (mm/s)', '전류 (A)']

stats = df.groupby('label_name')[features].agg(['mean', 'std', 'min', 'max'])
stats.columns = ['_'.join(c) for c in stats.columns]
stats.round(3)

## 2. 분포 비교 — 히스토그램

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'정상': '#3B82F6', '이상': '#EF4444'}

for ax, feat, label in zip(axes, features, labels):
    for grp, color in colors.items():
        subset = df[df.label_name == grp][feat]
        ax.hist(subset, bins=50, alpha=0.6, color=color, label=grp, density=True)
    ax.set_title(label, fontsize=13)
    ax.set_xlabel(label)
    ax.set_ylabel('밀도')
    ax.legend()

plt.suptitle('정상 vs 이상 데이터 분포 비교', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../data/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ 이상 데이터는 정상 범위를 크게 벗어남을 확인')

## 3. 산점도 — 2D 피처 공간에서 이상 패턴

In [ ]:
pairs = [
    ('temperature', 'vibration', '온도', '진동'),
    ('temperature', 'current',   '온도', '전류'),
    ('vibration',   'current',   '진동', '전류'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (x, y, xl, yl) in zip(axes, pairs):
    for grp, color, alpha in [('정상', '#3B82F6', 0.3), ('이상', '#EF4444', 0.8)]:
        sub = df[df.label_name == grp]
        ax.scatter(sub[x], sub[y], c=color, alpha=alpha, s=10, label=grp)
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.set_title(f'{xl} vs {yl}')
    ax.legend(markerscale=3)

plt.suptitle('피처 쌍 산점도 — 이상 클러스터 확인', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/eda_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ 이상 데이터는 3가지 서로 다른 이상 패턴으로 분리됨')

## 4. 박스플롯 — 이상치 범위 정량화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, feat, label in zip(axes, features, labels):
    data = [df[df.label==0][feat].values, df[df.label==1][feat].values]
    bp = ax.boxplot(data, labels=['정상', '이상'], patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('#93C5FD')
    bp['boxes'][1].set_facecolor('#FCA5A5')
    ax.set_title(label, fontsize=12)
    ax.set_ylabel(label)

plt.suptitle('센서별 정상/이상 박스플롯', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. 상관관계 히트맵

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, grp, title in [(axes[0], 0, '정상 데이터'), (axes[1], 1, '이상 데이터')]:
    corr = df[df.label == grp][features].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                xticklabels=labels, yticklabels=labels,
                vmin=-1, vmax=1, ax=ax)
    ax.set_title(title, fontsize=12)

plt.suptitle('피처 간 상관관계', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('→ 정상 데이터에서 피처 간 상관관계가 낮음 (독립적 측정값)')

## 6. 시계열 샘플 — 이상 발생 구간 시각화

In [ ]:
# 처음 500건만 시각화 (이상이 섞여 있는 구간 포함)
sample = df.head(500).reset_index(drop=True)

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
colors_ts = sample['label'].map({0: '#3B82F6', 1: '#EF4444'})

for ax, feat, label in zip(axes, features, labels):
    ax.plot(sample.index, sample[feat], color='#CBD5E1', linewidth=0.8, zorder=1)
    ax.scatter(sample.index, sample[feat], c=colors_ts, s=8, zorder=2)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('샘플 인덱스')

from matplotlib.patches import Patch
legend = [Patch(color='#3B82F6', label='정상'), Patch(color='#EF4444', label='이상')]
axes[0].legend(handles=legend, loc='upper right')
plt.suptitle('센서 시계열 — 이상 발생 구간 (빨간점)', fontsize=14)
plt.tight_layout()
plt.savefig('../data/eda_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. EDA 요약

| 이상 유형 | 온도 | 진동 | 전류 | 비율 |
|-----------|------|------|------|---------|
| 과열 | **~95°C** (정상 70°C) | 정상 | 정상 | 33% |
| 과진동 | 정상 | **~2.5mm/s** (정상 0.5) | 정상 | 33% |
| 전류 스파이크 | 정상 | 정상 | **~30A** (정상 12A) | 33% |

**결론:**
- 세 채널이 각각 독립적으로 이상을 나타냄 → 단변량 임계값으로도 일부 탐지 가능
- 그러나 **복합 이상** (e.g. 온도 약간 높음 + 진동 약간 높음)은 단순 임계값으로 탐지 어려움
- → IsolationForest로 **다변량 비선형 이상탐지** 적용 타당